In [42]:
#binary classification 0 - neg , 1 - pos
# !pip install -U datasets transformers

from datasets import load_dataset
from transformers import AutoTokenizer

data = load_dataset("stanfordnlp/imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [43]:
data['train'].shape , data['test'].shape

((25000, 2), (25000, 2))

In [44]:
sample =  data['train'][0]['text']
enc = tokenizer(sample, truncation=True, max_length=256)
# print(enc.keys()) #input_ids , token_type_ids
print(enc['input_ids'])
text = tokenizer.convert_ids_to_tokens(enc['input_ids'])
print(text)
print("pad id:", tokenizer.pad_token_id, "| vocab size:", tokenizer.vocab_size)

[101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 2107, 2004, 1996, 5148, 2162, 1998, 2679, 3314, 1999, 1996, 2142, 2163, 1012, 1999, 2090, 4851, 8801, 1998, 6623, 7939, 4697, 3619, 1997, 8947, 2055, 2037, 10740, 2006, 4331, 1010, 2016, 2038, 3348, 2007, 2014, 3689, 3836, 1010, 19846

In [45]:
def tokenizer_data(data):
  return tokenizer(data['text'], truncation=True, max_length=256)

MAX_LEN = 256
tokenized_train_data = data['train'].map(tokenizer_data, batched=True)
tokenized_test_data = data['test'].map(tokenizer_data, batched=True)

In [46]:
tokenized_train_data

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [47]:
cols = ["input_ids", "attention_mask", "label"]
for ds in (tokenized_train_data, tokenized_test_data):
    ds.set_format(type="torch", columns=cols)  #convert each to torch tensor

In [48]:
#tokenization and tensor dtype cpnversion is over
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

batch_size = 32
train_dataloader = DataLoader(tokenized_train_data, shuffle=True, batch_size= batch_size, collate_fn=DataCollatorWithPadding(tokenizer=tokenizer)) # pads each batch to its longest item with tokenizer.pad token
test_dataloader = DataLoader(tokenized_test_data, batch_size= batch_size, collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))

In [49]:
len(train_dataloader) , len(test_dataloader)  # 782 * 32 each.

(782, 782)

In [50]:
# next(iter(train_dataloader)) # input_ids , attention_mask, labels

In [51]:
# import torch.nn as nn
# import torch

# class LinearNN(nn.Module):
#   def __init__(self):
#     super().__init__()
#     self.embed = nn.Embedding(tokenizer.vocab_size, 768)
#     self.embed_dropout = nn.Dropout(0.4)  # Prevent embedding memorization
#     self.l1 = nn.Linear(768, 128)
#     self.batchnorm = nn.BatchNorm1d(128)
#     self.Relu = nn.ReLU()
#     self.dropout = nn.Dropout(0.4)        # Robust feature regularization
#     self.l2 = nn.Linear(128,1)

#   def forward(self, x , mask):
#     x = self.embed(x)
#     x = self.embed_dropout(x)  # Drop out individual tokens

#     mask = mask.unsqueeze(-1).float()
#     # Average token embeddings safely ignoring padding tokens
#     x = torch.sum(x * mask, dim=1) / torch.sum(mask, dim=1)
#     x = self.l1(x)
#     x = self.batchnorm(x)
#     x = self.Relu(x)
#     x = self.dropout(x)
#     return self.l2(x)

# torch.manual_seed(42)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_1 = LinearNN().to(device)
# print(f"Re-initialized model on: {device}")

In [52]:
# import torch.nn as nn
# import torch

# class RNN(nn.Module):
#   def __init__(self):
#     super().__init__()
#     self.embed = nn.Embedding(tokenizer.vocab_size, 768)
#     self.embed_dropout = nn.Dropout(0.4)  # Prevent embedding memorization
#     self.Rnn = nn.RNN(768, 128, batch_first=True, bidirectional=True, num_layers= 3)
#     self.batchnorm = nn.BatchNorm1d(128)
#     self.Relu = nn.ReLU()
#     self.dropout = nn.Dropout(0.4)        # Robust feature regularization
#     self.l2 = nn.Linear(128,1)

#   def forward(self, x , mask):
#     x = self.embed(x)                     # [batch_size, seq_len, 768]
#     x = self.embed_dropout(x)

#     # Pass the 3D tensor to the RNN
#     # out shape: [batch_size, seq_len, 128]
#     # h_n shape: [1, batch_size, 128]
#     out, h_n = self.Rnn(x)
#     # print(h_n.shape)
#     # Use the last hidden state of the sequence
#     x = h_n[-1]                        # [batch_size, 128]

#     x = self.batchnorm(x)
#     x = self.Relu(x)
#     x = self.dropout(x)
#     return self.l2(x)

# torch.manual_seed(42)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_1 = RNN().to(device)
# print(f"Re-initialized model on: {device}")

In [53]:
# #Transformer encoder as classifier from scratch
# import torch.nn as nn
# import torch

# class TransformerClassifier(nn.Module):
#   def __init__(self):
#     super().__init__()
#     self.proj = nn.Embedding(tokenizer.vocab_size, 768)
#     self.positinal_embed = nn.Embedding(MAX_LEN, 768)
#     self.dropout = nn.Dropout(0.4)

#     encoder_layer = nn.TransformerEncoderLayer(d_model=768, nhead=8)
#     self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=6)
#     self.l1 = nn.Linear(768, 128)
#     self.batchnorm = nn.BatchNorm1d(128)
#     self.Relu = nn.ReLU()
#     self.dropout = nn.Dropout(0.4)        # Robust feature regularization
#     self.l2 = nn.Linear(128,1)

#   def forward(self, x , mask):
#     x = self.proj(x)
#     x = x + self.positinal_embed.weight[:x.size(1), :]
#     x = self.dropout(x)
#     x = self.transformer_encoder(x)
#     x = x.mean(dim=1)
#     x = self.l1(x)
#     x = self.batchnorm(x)
#     x = self.Relu(x)
#     x = self.dropout(x)
#     return self.l2(x)

# torch.manual_seed(42)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_1 = TransformerClassifier().to(device)
# print(f"Re-initialized model on: {device}")

In [54]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class FFN(nn.Module):
  def __init__(self,h_s):
    super().__init__()
    self.l1 = nn.Linear(h_s, 4*h_s)
    self.batchnorm = nn.LayerNorm(h_s)
    self.Relu = nn.ReLU()
    self.dropout = nn.Dropout(0.3)
    self.l2 = nn.Linear(4*h_s, h_s)

  def forward(self, x):
    x = self.l1(self.batchnorm(x))
    x = self.dropout(x)
    return self.l2(self.Relu(x))


#for all heads input is x only , only key , val ,q matrices changes (multiple) and it is been provent hat each head will learn some thing / attention like nouns, puntuatuions...
class MultiHeadSelfAttention(nn.Module):
  def __init__(self, h_s, num_heads=8):
    super().__init__()
    assert h_s % num_heads == 0, "Hidden size must be divisible by the number of heads"
    self.h_s = h_s
    self.num_heads = num_heads
    self.depth = h_s // num_heads

    self.k = nn.Linear(h_s, h_s)
    self.v = nn.Linear(h_s, h_s)
    self.q = nn.Linear(h_s, h_s)

    self.out_proj = nn.Linear(h_s, h_s)
    self.dropout = nn.Dropout(0.4)
    self.batchnorm = nn.LayerNorm(h_s)

  def split_heads(self, x, batch_size):
    # Split the last dimension into (num_heads, depth)
    # Transpose to shape: (batch_size, num_heads, seq_len, depth)
    x = x.view(batch_size, -1, self.num_heads, self.depth)
    return x.transpose(1, 2)

  def forward(self, x, mask):
    batch_size = x.size(0)
    x = self.batchnorm(x)

    key = self.split_heads(self.k(x), batch_size)     # [B, num_heads, seq_len, depth]
    val = self.split_heads(self.v(x), batch_size)     # [B, num_heads, seq_len, depth]
    query = self.split_heads(self.q(x), batch_size)   # [B, num_heads, seq_len, depth]

    # Calculate attention scores
    scores = torch.matmul(query, key.transpose(-2, -1)) # [B, num_heads, seq_len, seq_len]
    scores = scores / (self.depth ** 0.5)

    # Apply mask (broadcasts automatically across heads)
    scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = torch.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)

    # Weighted sum
    context = torch.matmul(attention_weights, val)    # [B, num_heads, seq_len, depth]

    # Concatenate heads back
    context = context.transpose(1, 2).contiguous()    # [B, seq_len, num_heads, depth]
    context = context.view(batch_size, -1, self.h_s)  # [B, seq_len, h_s]

    return self.out_proj(context)


class Encoder(nn.Module):
   def __init__(self, h_s, num_heads=8):
    super().__init__()
    self.attention = MultiHeadSelfAttention(h_s, num_heads)
    self.fnn = FFN(h_s)

   def forward(self, x, mask):
    x = x + self.attention(x, mask)
    x = x + self.fnn(x)
    return x


class Transformer(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=768, n_layers=6, num_heads=8):
      super().__init__()
      self.proj = nn.Embedding(vocab_size, d_model)
      self.dropout = nn.Dropout(0.3)
      self.pos = nn.Embedding(max_len, d_model)
      self.l1 = nn.Linear(d_model, 256)
      self.Relu = nn.ReLU()
      self.l2 = nn.Linear(256, 1)
      self.encoder_layers = nn.ModuleList([Encoder(d_model, num_heads) for _ in range(n_layers)])

    def forward(self, x, mask):
     seq_len = x.size(1)
     mask = mask.unsqueeze(1).unsqueeze(2) # adjust mask dimensions for Multi-Head compatibility [B, 1, 1, seq_len]
     x = self.proj(x)

     pos_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)
     x = x + self.pos(pos_ids)

     for encoder in self.encoder_layers:
      x = encoder(x, mask)
     x = x.mean(dim=1)
     x = self.dropout(x)
     x = self.Relu(self.l1(x))
     return self.l2(x)


device = "cuda" if torch.cuda.is_available() else "cpu"
model_1 = Transformer(tokenizer.vocab_size, MAX_LEN, num_heads=8).to(device)
print(model_1)

rand_input = torch.randint(0, tokenizer.vocab_size, (32, 256), dtype=torch.long).to(device)
rand_mask = torch.ones(32, 256).to(device)
print(model_1(rand_input, rand_mask).shape)

Transformer(
  (proj): Embedding(30522, 768)
  (dropout): Dropout(p=0.3, inplace=False)
  (pos): Embedding(256, 768)
  (l1): Linear(in_features=768, out_features=256, bias=True)
  (Relu): ReLU()
  (l2): Linear(in_features=256, out_features=1, bias=True)
  (encoder_layers): ModuleList(
    (0-5): 6 x Encoder(
      (attention): MultiHeadSelfAttention(
        (k): Linear(in_features=768, out_features=768, bias=True)
        (v): Linear(in_features=768, out_features=768, bias=True)
        (q): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.4, inplace=False)
        (batchnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (fnn): FFN(
        (l1): Linear(in_features=768, out_features=3072, bias=True)
        (batchnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (Relu): ReLU()
        (dropout): Dropout(p=0.3, inplace=False)
        (l2

In [55]:
# import torch.nn as nn
# import torch
# import torch.nn.functional as F

# class SelfAttention(nn.Module):          # Step 1 + 2 combined
#     def __init__(self, d_model, dropout):
#         super().__init__()
#         # q_proj, k_proj, v_proj, , dropout
#         self.k_proj = nn.Linear(d_model, d_model)
#         self.q_proj = nn.Linear(d_model, d_model)
#         self.v_proj = nn.Linear(d_model, d_model)
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, x, mask):
#         # project  → scores → mask → softmax → weighted sum → merge → out_proj
#         Q = self.q_proj(x)
#         K = self.k_proj(x)
#         V = self.v_proj(x)
#         scores = torch.matmul(Q, K.permute(0,2,1)) / (K.size(-1) ** 0.5)
#         scores = scores.masked_fill(mask == 0, -1e9)
#         weights = F.softmax(scores, dim=-1)
#         weights = self.dropout(weights)
#         return torch.matmul(weights, V)

# class FeedForward(nn.Module):            # Step 3
#     def __init__(self, d_model, d_ff, dropout):
#       super().__init__()
#       self.linear = nn.Linear(d_model ,d_ff)
#       self.Relu = nn.ReLU()
#       self.dropout = nn.Dropout(dropout)
#       self.linear2 = nn.Linear(d_ff, d_model)
#     def forward(self, x):
#       x = self.linear(x)
#       x = self.Relu(x)
#       x = self.dropout(x)
#       x = self.linear2(x)
#       return x

# class EncoderBlock(nn.Module):           # Step 4
#     def __init__(self, d_model, d_ff, dropout):
#         # norm1, attn, norm2, ffn, dropout
#         super().__init__()
#         self.norm1 = nn.LayerNorm(d_model)
#         self.norm2 = nn.LayerNorm(d_model)
#         self.attn = SelfAttention(d_model, dropout)
#         self.ffn = FeedForward(d_model, d_ff, dropout)
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, x, mask):
#         x = x + self.dropout(self.attn(self.norm1(x), mask))
#         x = x + self.dropout(self.ffn(self.norm2(x)))
#         return x

# class TransformerClassifier(nn.Module):  # Steps 5 + 6
#     def __init__(self, vocab_size, max_len, d_model=768, n_layers=6):
#         super().__init__()
#       # tok_emb, pos_emb, dropout,
#         self.token_emb = nn.Embedding(vocab_size, d_model)
#         self.pos_emb = nn.Embedding(max_len, d_model)
#         self.dropout = nn.Dropout(0.4)
#         self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, d_model*4, 0.4) for _ in range(n_layers)])
#         self.l1 = nn.Linear(d_model, 128)
#         self.batchnorm = nn.BatchNorm1d(128)
#         self.Relu = nn.ReLU()
#         self.dropout = nn.Dropout(0.4)        # Robust feature regularization
#         self.l2 = nn.Linear(128,1)

#     def forward(self, x, mask):
#         # Create sequence mask for self-attention
#         # mask: [batch_size, seq_len] -> [batch_size, 1, seq_len] for proper broadcasting with scores matrix
#         mask = mask.unsqueeze(1)
#         x = self.token_emb(x)
#         x = x + self.pos_emb.weight[:x.size(1), :]
#         x = self.dropout(x)
#         for block in self.encoder_blocks:
#             x = block(x, mask)
#         x = x.mean(dim=1)  #kep dim is not set here
#         x = self.l1(x)
#         x = self.batchnorm(x)
#         x = self.Relu(x)
#         x = self.dropout(x)
#         return self.l2(x)

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model_1 = TransformerClassifier(tokenizer.vocab_size, MAX_LEN).to(device)
# print(model_1)

# # Correctly generate random integer token IDs
# rand_input = torch.randint(0, tokenizer.vocab_size, (32, 256), dtype=torch.long).to(device)
# rand_mask = torch.ones(32, 256).to(device)
# print(model_1(rand_input, rand_mask).shape)

In [56]:
#print no of param
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"The model has {count_parameters(model_1):,} trainable parameters")

The model has 66,361,857 trainable parameters


In [ ]:
model_1.train()
epochs = 5
optimizer = torch.optim.AdamW(model_1.parameters() , lr = 0.0001 , weight_decay = 0.01)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(epochs):
  epoch_loss = 0
  epoch_correct = 0
  total_samples = 0

  for idx , batch in enumerate(train_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}
    current_batch_size = len(batch['labels'])

    y_logit =  model_1(batch['input_ids'], batch['attention_mask'])
    y_pred = torch.sigmoid(y_logit)
    y_pred = torch.round(y_pred)

    batch_correct = (y_pred == batch['labels'].unsqueeze(1)).sum().item()
    epoch_correct += batch_correct
    total_samples += current_batch_size

    loss = loss_fn(y_logit, batch['labels'].unsqueeze(1).float())
    optimizer.zero_grad()
    epoch_loss += loss.item() * current_batch_size
    loss.backward()
    optimizer.step()

    if idx % 10 == 0:
      print(f'epoch {epoch+1} | batch {idx} | batch loss: {loss.item():.4f} | batch Accuracy: {batch_correct / current_batch_size:.4f}')

  print('epoch loss is ', epoch_loss / total_samples)
  print('epoch Accuracy is' , epoch_correct / total_samples)

epoch 1 | batch 0 | batch loss: 0.7071 | batch Accuracy: 0.4062
epoch 1 | batch 10 | batch loss: 0.7013 | batch Accuracy: 0.5000
epoch 1 | batch 20 | batch loss: 0.5979 | batch Accuracy: 0.6875
epoch 1 | batch 30 | batch loss: 0.7161 | batch Accuracy: 0.4375
epoch 1 | batch 40 | batch loss: 0.6437 | batch Accuracy: 0.7188
epoch 1 | batch 50 | batch loss: 0.7034 | batch Accuracy: 0.4375
epoch 1 | batch 60 | batch loss: 0.7584 | batch Accuracy: 0.4062
epoch 1 | batch 70 | batch loss: 0.6754 | batch Accuracy: 0.5938
epoch 1 | batch 80 | batch loss: 0.7273 | batch Accuracy: 0.4375
epoch 1 | batch 90 | batch loss: 0.7017 | batch Accuracy: 0.5000
epoch 1 | batch 100 | batch loss: 0.7285 | batch Accuracy: 0.4375
epoch 1 | batch 110 | batch loss: 0.7085 | batch Accuracy: 0.4375
epoch 1 | batch 120 | batch loss: 0.7269 | batch Accuracy: 0.3438
epoch 1 | batch 130 | batch loss: 0.6487 | batch Accuracy: 0.5938
epoch 1 | batch 140 | batch loss: 0.5935 | batch Accuracy: 0.7188
epoch 1 | batch 150 |

In [ ]:
#on test set
model_1.eval()
test_loss = 0
test_correct = 0
total_test_samples = 0

with torch.inference_mode():
  for batch in test_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    current_batch_size = len(batch['labels'])

    y_logit = model_1(batch['input_ids'], batch['attention_mask'])
    y_pred = torch.round(torch.sigmoid(y_logit))

    loss = loss_fn(y_logit, batch['labels'].unsqueeze(1).float())
    test_loss += loss.item() * current_batch_size

    batch_correct = (y_pred == batch['labels'].unsqueeze(1)).sum().item()
    test_correct += batch_correct
    total_test_samples += current_batch_size

print('Test loss:', test_loss / total_test_samples)
print('Test Accuracy:', test_correct / total_test_samples)

In [ ]:
review = "This is a great movie actually"
tokens = tokenizer(review, truncation=True, max_length=256)
# print(tokens)
input_ids = torch.tensor(tokens['input_ids']).to(device)
attention_mask = torch.tensor(tokens['attention_mask']).to(device)

with torch.inference_mode():
  input_ids = input_ids.unsqueeze(0)
  attention_mask = attention_mask.unsqueeze(0)
  output = model_1(input_ids, attention_mask)
  y_pred = torch.sigmoid(output)
  y_pred = torch.round(y_pred)

print(y_pred)
